In [1]:
import torch
from datasets import load_dataset, concatenate_datasets
from transformers import AutoModelForCausalLM, AutoTokenizer
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import QuantizationModifier
from compressed_tensors.offload import dispatch_model

ModuleNotFoundError: No module named 'datasets'

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
import os
print(os.environ["HF_TOKEN"])

In [ ]:
MODEL_ID = "zai-org/GLM-5.1"
SAVE_DIR = "/workspace/GLM-5.1-NVFP4"
MAX_SEQUENCE_LENGTH = 2048

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype="auto",
    device_map=None,
    cache_dir="/workspace",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir="/workspace")

In [ ]:


# ds = load_dataset("HuggingFaceH4/ultrachat_200k", split=f"train_sft[:{NUM_CALIBRATION_SAMPLES}]")
# ds = ds.shuffle(seed=42)

# https://huggingface.co/datasets/tejeshbhalla/tool_calling/viewer/default/train?row=1
# https://huggingface.co/datasets/AymanTarig/function-calling-v0.2-with-r1-cot

NUM_CALIBRATION_SAMPLES = 1

code = load_dataset("nvidia/Nemotron-Post-Training-Dataset-v2", split="code", streaming=True, cache_dir="/workspace/hf_cache")
math = load_dataset("nvidia/Nemotron-Post-Training-Dataset-v2", split="math", streaming=True, cache_dir="/workspace/hf_cache")

code_samples = list(code.take(1))
math_samples = list(math.take(2))

from datasets import Dataset
# ds = Dataset.from_list(code_samples + math_samples).shuffle(seed=1337)
ds = Dataset.from_list(code_samples).shuffle(seed=1337)

def preprocess(example):
    return {"text": tokenizer.apply_chat_template(example["messages"], tokenize=False)}

ds = ds.map(preprocess)

def tokenize(sample):
    return tokenizer(
        sample["text"],
        padding=False,
        max_length=MAX_SEQUENCE_LENGTH,
        truncation=True,
        add_special_tokens=False,
    )

ds = ds.map(tokenize, remove_columns=ds.column_names)

In [ ]:
recipe = QuantizationModifier(
    targets="Linear",
    scheme="NVFP4",
    ignore=[
        "re:.*lm_head",
        "re:.*mlp.gate$",
        "re:.*embed_tokens$",
        "re:.*shared_expert_gate$",
    ],
)

In [ ]:
oneshot(
    model=model,
    recipe=recipe,
    dataset=ds,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    num_calibration_samples=NUM_CALIBRATION_SAMPLES,
    moe_calibrate_all_experts=True,
)

In [ ]:
model.generation_config.do_sample = True
model.save_pretrained(SAVE_DIR, save_compressed=True)
tokenizer.save_pretrained(SAVE_DIR)